# Wildfire Risk Model v3 (district x day, ECMWF IFS weather)

Trains `wildfire_xgb_v3` on the Kaggle dataset `shaikhfardin05/india-district-wildfire-v3`, produced by
`ml/pipeline/prepare_ifs_weather.py` + `ml/pipeline/build_wildfire_dataset.py` (NASA FIRMS VIIRS NOAA-20 SP labels +
ECMWF IFS HRES 9 km weather, the model live serving receives via Open-Meteo + project FWI code).

The notebook only calls `ml/train/train_wildfire_v3.py` (shipped inside the dataset), so Kaggle and local runs execute
identical code. Split: train 2019-2023, validation 2024, test 2025-01-01..2025-12-31 (scored once). Baselines
(FWI-only, climatology, persistence) are computed on the same test rows. XGBoost runs on the GPU (`device=cuda`).
The deployed `wildfire_xgb_v2` is scored on the same 2025 rows (IFS weather and NASA POWER weather).
Outputs land in `/kaggle/working`.

In [ ]:
import glob, os, subprocess, sys

root = os.path.dirname(glob.glob("/kaggle/input/**/wildfire_dataset.parquet", recursive=True)[0])
print("dataset root:", root)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
cmd = [
    sys.executable,
    f"{root}/ml/train/train_wildfire_v3.py",
    "--dataset",
    f"{root}/wildfire_dataset.parquet",
    "--climatology",
    f"{root}/climatology.json",
    "--out-dir",
    "/kaggle/working",
    "--device",
    "cuda",
    "--v2-model",
    f"{root}/v2/wildfire_xgb_v2.json",
    "--v2-metadata",
    f"{root}/v2/wildfire_xgb_v2_metadata.json",
    "--power-dataset",
    f"{root}/power/wildfire_dataset_power.parquet",
]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-5000:])
r.check_returncode()

In [ ]:
import json

ev = json.load(open("/kaggle/working/wildfire_v3_evaluation.json"))
print(json.dumps({k: ev[k] for k in ("test", "v2_vs_v3_same_2025_rows") if k in ev}, indent=2))